# Modelling antenna beams

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/processing_functions_tutorials/simulation/antenna_beams.ipynb)

Port of the SIRIUS `antenna_beams` notebook.  The simulator supports three kinds of antenna
(voltage) beam models; this notebook evaluates each of them into **Jones beam images**
``JONES[parallactic_angle, frequency, polarization, l, m]`` and forms **Mueller matrices**:

1. analytic Airy disks (CASA ``PBMath`` parameters),
2. CASA ``PBMath1DPoly`` beam polynomials (EVLA bands),
3. Zernike aperture-coefficient models (EVLA and MeerKAT, Sekhar et al. 2022) which include
   polarization leakage and rotate with the parallactic angle.

---
## API


In [ ]:
from astroviper.processing_functions.simulation import (
    evaluate_beam_models,
    make_airy_jones_beam,
    make_mueller_matrix,
    make_polynomial_jones_beam,
    make_zernike_jones_beam,
)

make_zernike_jones_beam?

## Install AstroVIPER

In [ ]:
import os
from importlib.metadata import version

try:
    import astroviper  # noqa: F401

    print("Using astroviper version", version("astroviper"))
except ImportError:
    os.system("pip install --upgrade astroviper")
    import astroviper  # noqa: F401

    print("Installed astroviper version", version("astroviper"))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from astropy.coordinates import SkyCoord

xr.set_options(display_style="html")
ARCSEC_TO_RAD = np.pi / (180 * 3600)

In [ ]:
import sys

sys.path.insert(0, ".")  # beam_plotting_helpers.py lives next to this notebook
from beam_plotting_helpers import display_jones, display_mueller

## 1. Airy disk

In [ ]:
from astroviper.utils.beam_models import AIRY_DISK_MODELS, airy_disk_model

print(AIRY_DISK_MODELS)
jones_airy = make_airy_jones_beam(
    airy_disk_model("alma"), frequency=[90e9], beam_params={"image_size": [256, 256]}
)
jones_airy

In [ ]:
display_jones(jones_airy, units="arcmin")
plt.show()

## 2. EVLA beam polynomials

In [ ]:
from astroviper.utils.beam_models import read_beam_polynomial_coefficients

bpc_xds = read_beam_polynomial_coefficients("EVLA_")
s_band = bpc_xds.sel(frequency=bpc_xds.frequency[bpc_xds.band == "S"])
jones_poly = make_polynomial_jones_beam(
    s_band, frequency=[2.5e9, 3.5e9], beam_params={"image_size": [256, 256]}
)
display_jones(jones_poly, frequency=0, units="arcmin")
plt.show()

## 3. Zernike aperture coefficients

The aperture illumination of each Jones element is built from 66 Zernike terms on a grid rotated
by the parallactic angle, masked to the dish and inverse Fourier transformed.

In [ ]:
from astroviper.utils.beam_models import (
    list_aperture_polynomial_coefficient_models,
    read_aperture_polynomial_coefficients,
)

print(list_aperture_polynomial_coefficient_models())
zpc_xds = read_aperture_polynomial_coefficients("EVLA_avg_zcoeffs_SBand_lookup")
zpc_xds

In [ ]:
parallactic_angles = np.linspace(0, np.pi / 2, 5)
beam_params = {"image_size": [256, 256], "mueller_selection": np.arange(16)}
jones_zernike = make_zernike_jones_beam(
    zpc_xds, parallactic_angles, frequency=[3.0e9], beam_params=beam_params
)
jones_zernike

In [ ]:
display_jones(jones_zernike, parallactic_angle=0, val_type="abs", units="arcmin")
plt.show()
display_jones(jones_zernike, parallactic_angle=2, val_type="phase", units="arcmin")
plt.show()

## Mueller matrix

$M = J_1 \otimes J_2^*$ for the selected elements (here all 16).

In [ ]:
mueller = make_mueller_matrix(
    jones_zernike, jones_zernike, mueller_selection=np.arange(16)
)
display_mueller(mueller, parallactic_angle=0, val_type="abs", units="arcmin")
plt.show()

## Evaluating models for an observation

``evaluate_beam_models`` is what the simulator calls: it computes the parallactic angles of the
observation, picks a representative subset (``pa_radius``) and evaluates every Zernike model on it.

In [ ]:
from astroviper.utils.telescope_layout import observatory_position

time = np.array(
    [
        "2019-10-03T16:00:00.000",
        "2019-10-03T18:00:00.000",
        "2019-10-03T20:00:00.000",
        "2019-10-03T22:00:00.000",
    ]
)
phase_center = SkyCoord(ra="19h59m28.5s", dec="+40d44m01.5s", frame="icrs")
models, parallactic_angle = evaluate_beam_models(
    [zpc_xds, airy_disk_model("vla")],
    time,
    np.array([3.0e9]),
    np.array([[phase_center.ra.rad, phase_center.dec.rad]]),
    observatory_position("EVLA"),
    {"image_size": [128, 128], "pa_radius": 0.2},
)
print("parallactic angles [deg]:", parallactic_angle * 180 / np.pi)
print(
    "beam images evaluated at [deg]:", models[0].parallactic_angle.values * 180 / np.pi
)
print("second model stays analytic:", models[1])